# Seasonal evaluation report

Scores of the AI ensembles over the season in the config, against each truth. Run the stages first:

```bash
uv run scripts/run_season.py configs/summer2026_france.yaml --stage forecast   # GPU
uv run scripts/run_season.py configs/summer2026_france.yaml --stage score      # CPU
```

The notebook only reads the saved per-case statistics (`outputs/season/<name>/scores/`), so it runs in seconds and can be re-run while the season is still being forecast: it reports whatever is scored so far.

All scores are fair estimators (comparable across ensemble sizes) and are computed from summed statistics over the selected init times, never as averages of per-case scores. Lower is better for every score except the spread-skill ratio, which should be 1.

In [ ]:
# Parameters
CONFIG = "configs/summer2026_france.yaml"
OVERRIDES = {}          # e.g. {"period": {"season": "JJA", "year": 2025}}
LEADS_FOR_TABLES = [24, 72, 120]   # hours

In [ ]:
import os, sys
from pathlib import Path

# run from the repo root wherever the notebook lives
root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from weather_pipeline.season import config
from weather_pipeline.season.scoring import load_statistics, season_scores

cfg = config.load(CONFIG, **OVERRIDES)
print(cfg.describe())

stats = {truth: load_statistics(cfg, truth) for truth in cfg.truths}
scored = {truth: {m: int(s.sizes["init_time"]) for m, s in by_model.items()} for truth, by_model in stats.items()}
print("\nscored initialisations:", scored)
stats = {t: s for t, s in stats.items() if s}
assert stats, "Nothing scored yet: run --stage score first"

# the climatological baseline is shown only where it is the reference
# (skill and extremes); elsewhere it would dwarf the models
from weather_pipeline.season.scoring import BASELINE
models_only = {t: {m: s for m, s in bm.items() if m != BASELINE} for t, bm in stats.items()}


## Season scores

Each metric is finalised over all scored initialisations. `n_init` differing between models means the forecast stage has not finished one of them yet: compare models only over common dates (next cell).

In [ ]:
def common_inits(by_model):
    sets = [set(s["init_time"].values) for s in by_model.values()]
    return np.array(sorted(set.intersection(*sets)))

metrics = {truth: {m: season_scores(s, common_inits(by_model)) for m, s in by_model.items()}
           for truth, by_model in models_only.items()}
for truth, by_model in models_only.items():
    print(f"{truth}: {len(common_inits(by_model))} initialisation(s) common to all models")


In [ ]:
hours = lambda da: da["lead_time"].values / np.timedelta64(1, "h")
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
panels = [("crps", "CRPS (K)"), ("ens_mean_rmse", "ensemble-mean RMSE (K)"),
          ("spread_skill", "spread-skill ratio"), ("scrps", "scaled CRPS"),
          ("energy_score", "energy score (K)"), ("large_error_frac", "fraction of SE > 4 K²")]
for truth, by_model in metrics.items():
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
    for ax, (key, label) in zip(axes.flat, panels):
        for model, m in by_model.items():
            ax.plot(hours(m[key]), m[key], marker="o", ms=3, label=model)
        if key == "spread_skill":
            ax.axhline(1, color="k", lw=0.8, ls="--")
        ax.set_title(label); ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set_xlabel("lead time (h)")
    axes[0, 0].legend()
    fig.suptitle(f"{cfg.name}: verified against {truth}")
    fig.tight_layout()


## Table at selected lead times

In [ ]:
rows = []
for truth, by_model in metrics.items():
    for model, m in by_model.items():
        for h in LEADS_FOR_TABLES:
            lt = np.timedelta64(h, "h")
            if lt not in m["lead_time"].values:
                continue
            v = m.sel(lead_time=lt)
            rows.append({"truth": truth, "model": model, "lead (h)": h,
                         "CRPS": float(v.crps), "SCRPS": float(v.scrps), "DSS": float(v.dss),
                         "RMSE": float(v.ens_mean_rmse), "spread": float(v.spread),
                         "SSR": float(v.spread_skill), "ES": float(v.energy_score)})
pd.DataFrame(rows).set_index(["truth", "lead (h)", "model"]).sort_index().round(3)


## Calibration: PIT histograms

Flat = calibrated. U-shaped = too little spread (the truth often falls outside the ensemble); dome-shaped = too much; sloped = biased.

In [ ]:
for truth, by_model in metrics.items():
    fig, axes = plt.subplots(1, len(LEADS_FOR_TABLES), figsize=(4.5 * len(LEADS_FOR_TABLES), 3.2), sharey=True)
    for ax, h in zip(np.atleast_1d(axes), LEADS_FOR_TABLES):
        for model, m in by_model.items():
            lt = np.timedelta64(h, "h")
            if lt in m["lead_time"].values:
                pit = m["pit_hist"].sel(lead_time=lt)
                ax.step(np.arange(pit.size + 1) / pit.size, np.r_[pit.values, pit.values[-1]],
                        where="post", label=model)
        ax.axhline(1, color="k", lw=0.8, ls="--"); ax.set_title(f"+{h} h"); ax.set_xlabel("PIT")
    np.atleast_1d(axes)[0].set_ylabel("density"); np.atleast_1d(axes)[0].legend()
    fig.suptitle(f"PIT vs {truth}"); fig.tight_layout()


## Spread-skill by spread bin

Points on the diagonal mean that when the ensemble says it is uncertain, it is. With 10 members the sample spread is noisy, which flattens this diagram even for a calibrated ensemble (see `metrics.case_statistics`): compare models with each other rather than with the diagonal alone.

In [ ]:
for truth, by_model in metrics.items():
    fig, axes = plt.subplots(1, len(LEADS_FOR_TABLES), figsize=(4.5 * len(LEADS_FOR_TABLES), 3.8))
    for ax, h in zip(np.atleast_1d(axes), LEADS_FOR_TABLES):
        lt = np.timedelta64(h, "h")
        top = 0
        for model, m in by_model.items():
            if lt in m["lead_time"].values:
                sp, rm = m["binned_spread"].sel(lead_time=lt), m["binned_rmse"].sel(lead_time=lt)
                ok = np.isfinite(sp) & np.isfinite(rm)
                ax.plot(sp[ok], rm[ok], marker="o", label=model)
                top = max(top, float(np.nanmax([sp[ok].max(), rm[ok].max()])))
        ax.plot([0, top], [0, top], "k--", lw=0.8)
        ax.set_title(f"+{h} h"); ax.set_xlabel("spread (K)"); ax.set_ylabel("RMSE (K)")
    np.atleast_1d(axes)[0].legend(); fig.suptitle(f"binned spread-skill vs {truth}"); fig.tight_layout()


## Bias

Ensemble-mean minus truth, area-weighted over land. Positive = the forecast is too warm. A PIT histogram sloping down (truth often in the lower part of the ensemble) is the same signal seen through calibration.

In [ ]:
fig, axes = plt.subplots(1, len(metrics), figsize=(6.5 * len(metrics), 3.6), sharey=True)
for ax, (truth, by_model) in zip(np.atleast_1d(axes), metrics.items()):
    for model, m in by_model.items():
        ax.plot(hours(m["bias"]), m["bias"], marker="o", ms=3, label=model)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(f"bias vs {truth} (K)"); ax.set_xlabel("lead time (h)"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].legend(); fig.tight_layout()


## How much do the two truths disagree?

ERA5 minus the IFS analysis at the same valid times, over the same land cells. This is a floor under the errors measured against the IFS analysis for forecasts that start from ERA5: at short lead times they are still close to ERA5, so they inherit this difference. Differences concentrate where the 9 km IFS and the 31 km ERA5 see different terrain (mountains) and in the boundary layer's daily cycle.

In [ ]:
from weather_pipeline.regions import to_signed_lon
from weather_pipeline.season.scoring import truth_disagreement

def show_map(ax, da, title, cmap="viridis", vmin=None, vmax=None):
    lon = to_signed_lon(da["lon"].values)
    order = np.argsort(lon)
    im = ax.pcolormesh(lon[order], da["lat"].values, da.values[:, order],
                       cmap=cmap, vmin=vmin, vmax=vmax, shading="nearest")
    ax.set_title(title, fontsize=10); ax.set_aspect(1 / np.cos(np.deg2rad(46.5)))
    plt.colorbar(im, ax=ax, shrink=0.8)

if {"era5", "ifs_fc0"} <= set(cfg.truths):
    try:
        d = truth_disagreement(cfg, "era5", "ifs_fc0")
    except FileNotFoundError as e:
        d = None; print(e)
    if d is not None:
        by_hour = d[["mean_diff", "rms_diff"]].groupby("time.hour").mean()
        print(f"{d.attrs['n_times']} common valid times")
        print(by_hour.to_dataframe().round(3))
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        lim = float(np.nanmax(np.abs(d["map_mean_diff"])))
        show_map(axes[0], d["map_mean_diff"], "ERA5 - IFS analysis: mean (K)", "RdBu_r", -lim, lim)
        show_map(axes[1], d["map_rms_diff"], "ERA5 - IFS analysis: RMS (K)", "magma_r", 0)
        fig.tight_layout()


## Maps

CRPS and bias per land cell, for forecast day 1 (leads 6-24 h) and day 5 (102-120 h), over the common initialisations. Colour scales are shared across models so they can be compared directly.

In [ ]:
from weather_pipeline.season.scoring import season_maps

maps = {truth: {m: season_maps(s, common_inits(by_model)) for m, s in by_model.items()}
        for truth, by_model in models_only.items()}
days = [d for d in (1, 5) if all(d in mp["lead_day"].values for bm in maps.values() for mp in bm.values())]
for truth, by_model in maps.items():
    for key, cmap, centred in (("crps", "magma_r", False), ("bias", "RdBu_r", True)):
        vals = np.concatenate([mp[key].sel(lead_day=days).values.ravel() for mp in by_model.values()])
        lim = float(np.nanpercentile(np.abs(vals) if centred else vals, 99))
        fig, axes = plt.subplots(len(by_model), len(days), figsize=(5.2 * len(days), 3.8 * len(by_model)), squeeze=False)
        for r, (model, mp) in enumerate(by_model.items()):
            for c, day in enumerate(days):
                show_map(axes[r, c], mp[key].sel(lead_day=day), f"{model}, day {day}", cmap,
                         -lim if centred else 0, lim)
        fig.suptitle(f"{key} vs {truth} (K)"); fig.tight_layout()


## By elevation

Area-weighted scores per elevation band (IFS orography, same grid as the land mask). Your `findings.md` already found that spatial patterns over France are mostly elevation in disguise; this checks it for the ensembles, against both truths.

In [ ]:
from weather_pipeline.season.scoring import elevation, elevation_bands, land_weights, summed
from weather_pipeline.season.metrics import by_band

bands = elevation_bands(elevation(cfg))
w = land_weights(cfg)
rows = []
for truth, by_model in models_only.items():
    for model, s in by_model.items():
        b = by_band(summed(s, common_inits(by_model)), bands, w)
        for band in b["band"].values:
            for day in b["lead_day"].values:
                v = b.sel(band=band, lead_day=day)
                rows.append({"truth": truth, "model": model, "band": band, "day": int(day),
                             "cells": int(v.cells), "CRPS": float(v.crps), "bias": float(v.bias),
                             "RMSE": float(v.ens_mean_rmse), "SSR": float(v.spread_skill)})
by_elev = pd.DataFrame(rows)
fig, axes = plt.subplots(2, len(stats), figsize=(6.5 * len(stats), 6.5), squeeze=False, sharex=True)
for c, truth in enumerate(stats):
    sub = by_elev[(by_elev.truth == truth) & (by_elev.day.isin([1, 5]))]
    for (model, day), g in sub.groupby(["model", "day"]):
        style = "-" if day == 1 else "--"
        axes[0, c].plot(g.band, g.CRPS, style, marker="o", label=f"{model} day {day}")
        axes[1, c].plot(g.band, g.bias, style, marker="o", label=f"{model} day {day}")
    axes[0, c].set_title(f"CRPS by elevation vs {truth} (K)"); axes[1, c].set_title(f"bias by elevation vs {truth} (K)")
    axes[1, c].axhline(0, color="k", lw=0.8)
    for ax in axes[:, c]:
        ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=20)
axes[0, 0].legend(fontsize=8); fig.tight_layout()
by_elev[by_elev.day.isin([1, 5])].pivot_table(index=["truth", "band"], columns=["model", "day"],
                                              values="CRPS", sort=False).round(3)


## Skill against climatology

CRPS skill score, 1 − CRPS(model) / CRPS(climatology), over the same initialisations and lead times. 0 = no better than the ERA5 climatological ensemble (the same calendar day and hour in each of the climatology years), 1 = perfect. Needs `--stage climatology` before `--stage score`.

In [ ]:
from weather_pipeline.season.metrics import skill_score
from weather_pipeline.season.scoring import BASELINE

fig, axes = plt.subplots(1, len(stats), figsize=(6.5 * len(stats), 3.8), squeeze=False)
for ax, (truth, by_model) in zip(axes[0], stats.items()):
    if BASELINE not in by_model:
        ax.set_title(f"{truth}: no climatology scored yet"); continue
    common = common_inits(by_model)
    ref = season_scores(by_model[BASELINE], common)["crps"]
    for model, s in by_model.items():
        if model == BASELINE:
            continue
        ss = skill_score(season_scores(s, common)["crps"], ref)
        ax.plot(hours(ss), ss, marker="o", ms=3, label=model)
    ax.axhline(0, color="k", lw=0.8); ax.set_ylim(top=1)
    ax.set_title(f"CRPS skill vs climatology, {truth}"); ax.set_xlabel("lead time (h)"); ax.grid(alpha=0.3)
axes[0, 0].legend(); fig.tight_layout()


## Extremes

Per climatological threshold (ERA5 per cell, month and hour; quantile 1.0 = the record for that month and hour). All scores here use thresholds fixed in advance, so they are proper and avoid the forecaster's dilemma:

- **Brier score** for exceeding the threshold, and **threshold-weighted CRPS** (CRPS of the part of the distribution above it): lower is better.
- **Occurrence ratio**: observed exceedances / forecast-expected exceedances. Above 1 = the model under-forecasts the extreme.
- **Miss rate**: fraction of exceedances where the truth was above *every* member; at quantile 1.0, how often a record fell outside the whole ensemble.
- **CPIT KS**: distance from uniform of the conditional PIT of exceedances (0 = calibrated in the tail).

Shown for day 1 and day 5 leads (24 h and 120 h).

In [ ]:
ext_keys = [("brier", "Brier"), ("twcrps", "twCRPS (K)"), ("occurrence_ratio", "occurrence ratio"),
            ("miss_rate", "miss rate"), ("cpit_ks", "CPIT KS"), ("event_frac", "event frequency")]
for truth, by_model in stats.items():
    models = [m for m in by_model if m != BASELINE] + ([BASELINE] if BASELINE in by_model else [])
    if not all(int(by_model[m].attrs.get("has_thresholds", 0)) for m in models):
        print(f"{truth}: scored without thresholds; run --stage climatology, then --stage score"); continue
    common = common_inits(by_model)
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for k, model in enumerate(models):
        s = season_scores(by_model[model], common)
        for lt, style in ((np.timedelta64(24, "h"), "-"), (np.timedelta64(120, "h"), "--")):
            if lt not in s["lead_time"].values:
                continue
            v = s.sel(lead_time=lt)
            q = v["quantile"].values
            x = np.arange(len(q))                     # discrete levels, evenly spaced
            for ax, (key, label) in zip(axes.flat, ext_keys):
                ax.plot(x, v[key], style, marker="o", color=colors[k] if model != BASELINE else "grey",
                        label=f"{model} +{int(lt / np.timedelta64(1, 'h'))} h")
                ax.set_title(label); ax.grid(alpha=0.3)
    labels = ["record" if qq == 1 else f"{100 * qq:g}%" for qq in q]
    for ax in axes.flat:
        ax.set_xticks(np.arange(len(labels)), labels)
        ax.set_xlabel("climatological threshold")
    axes[0, 2].axhline(1, color="k", lw=0.8)
    axes[0, 0].legend(fontsize=7); fig.suptitle(f"extremes vs {truth}"); fig.tight_layout()


## Uncertainty

**Bands**: 90 % intervals from a paired block bootstrap over initialisation dates (blocks of 5 consecutive days, so the day-to-day persistence of weather regimes is respected; the same draws for every model, so differences compare identical cases).

**Diebold–Mariano**: for each pair of models and lead time, whether the CRPS difference is significant, after a Benjamini–Hochberg correction across lead times (5 % false discovery rate). With a season of strongly autocorrelated daily cases the test remains somewhat liberal (about 10 % false alarms at a nominal 5 % in synthetic tests): read borderline cases together with the bands.

In [ ]:
from itertools import combinations
from weather_pipeline.season.metrics import skill_score
from weather_pipeline.season.scoring import BASELINE
from weather_pipeline.season.significance import bootstrap, diebold_mariano, interval

N_BOOT = 1000
fig, axes = plt.subplots(2, len(stats), figsize=(6.5 * len(stats), 7), squeeze=False, sharex=True)
dm_rows = []
for c, (truth, by_model) in enumerate(stats.items()):
    reps = bootstrap(by_model, n_boot=N_BOOT)
    models = [m for m in by_model if m != BASELINE]
    for k, model in enumerate(models):
        crps = metrics[truth][model]["crps"] if model in metrics[truth] else season_scores(by_model[model], common_inits(by_model))["crps"]
        iv = interval(reps[model]["crps"])
        col = colors[k] if model != BASELINE else "grey"
        axes[0, c].plot(hours(crps), crps, color=col, marker="o", ms=3, label=model)
        axes[0, c].fill_between(hours(crps), iv["low"], iv["high"], color=col, alpha=0.2)
    axes[0, c].set_title(f"CRPS with 90 % band, {truth} (K)"); axes[0, c].grid(alpha=0.3)
    if BASELINE in by_model:
        ref = reps[BASELINE]["crps"]
        for k, model in enumerate(models):
            ss_rep = skill_score(reps[model]["crps"], ref)
            iv = interval(ss_rep)
            ss = skill_score(season_scores(by_model[model], common_inits(by_model))["crps"],
                             season_scores(by_model[BASELINE], common_inits(by_model))["crps"])
            axes[1, c].plot(hours(ss), ss, color=colors[k], marker="o", ms=3, label=model)
            axes[1, c].fill_between(hours(ss), iv["low"], iv["high"], color=colors[k], alpha=0.2)
        axes[1, c].axhline(0, color="k", lw=0.8)
    axes[1, c].set_title(f"CRPS skill vs climatology, 90 % band, {truth}"); axes[1, c].grid(alpha=0.3)
    axes[1, c].set_xlabel("lead time (h)")
    for a, b in combinations(models, 2):
        dm = diebold_mariano(by_model[a], by_model[b])
        for lt in dm["lead_time"].values:
            v = dm.sel(lead_time=lt)
            dm_rows.append({"truth": truth, "pair": f"{a} − {b}", "lead (h)": int(lt / np.timedelta64(1, "h")),
                            "CRPS diff (K)": float(v.mean_diff), "p": float(v.p_value),
                            "significant": bool(v.significant)})
axes[0, 0].legend(fontsize=8); axes[1, 0].legend(fontsize=8); fig.tight_layout()

dm_table = pd.DataFrame(dm_rows)
if len(dm_table):
    show = dm_table.assign(result=lambda t: np.where(~t.significant, "·",
                                                     np.where(t["CRPS diff (K)"] < 0, "first better", "second better")))
    display(show.pivot_table(index=["truth", "pair"], columns="lead (h)", values="CRPS diff (K)").round(3))
    display(show.pivot_table(index=["truth", "pair"], columns="lead (h)", values="result", aggfunc="first"))


## Coming next

Station verification (Météo-France), and the AIFS regridding check.